In [ ]:
import io
from datetime import datetime, timedelta
import math
from operator import itemgetter
from pathlib import Path
import zipfile

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import requests

from scipy.stats import gaussian_kde
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

In [ ]:
UCI_DATA_URL = "https://archive.ics.uci.edu/static/public/321/electricityloaddiagrams20112014.zip"
UCI_DATA_DIR = Path("../../data/uci")
UCI_RAW_TARGET_FILE_NAME = "LD2011_2014.txt"
UCI_DATA_PATH = UCI_DATA_DIR / UCI_RAW_TARGET_FILE_NAME

FREQUENCY_MINUTES = 15

## UCI Electricity Load Dataset

In [ ]:
# Download data

def load_uci_data(
    target_dir: str | Path = UCI_DATA_DIR,
    target_file: str | None = UCI_RAW_TARGET_FILE_NAME,
    url: str = UCI_DATA_URL
):
    response = requests.get(url)
    try: 
        response.raise_for_status()
    except requests.HTTPError as e:
        print(f"Failed to download from {url}. Reason: {e}")
        return

    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        try: 
            if target_file:
                z.extract(member=target_file, path=target_dir)
            else: 
                z.extractall(path=target_dir)
        except Exception as e:
            print(f"Failed to extract files. Reason: {e}")
            return


load_uci_data()

In [ ]:
# Load data as polars dataframe

UCI_DF = pl.read_csv(
    UCI_DATA_PATH,
    has_header=True,
    separator=";",
    decimal_comma=True,
    try_parse_dates=True,
    infer_schema_length=1_000_000
)

# First column should be timestamp column
UCI_DF = UCI_DF.rename({UCI_DF.columns[0]: "timestamp"}).sort(by="timestamp")

### Global EDA

In [ ]:
# Duplicate timestamps

duplicate_timestamps = UCI_DF.select(pl.col("timestamp")).is_duplicated().sum()
print(f"Number of duplicate timestamps found: {duplicate_timestamps}")

In [ ]:
# Timestamp for every interval

min_timestamp = UCI_DF.get_column("timestamp").min()
max_timestamp = UCI_DF.get_column("timestamp").max()

expected_timestamps = pl.datetime_range(
    start=min_timestamp,
    end=max_timestamp,
    interval=f"{FREQUENCY_MINUTES}m",
    closed="both",
    eager=True,
)

print("Expected number of timestamps: ", len(expected_timestamps))
print("Actual number of timestamps: ", len(UCI_DF))

In [ ]:
# Number of non-zero observations

non_zero_counts = (UCI_DF.select(pl.exclude("timestamp")) > 0).sum()

plt.hist(non_zero_counts.to_numpy().flatten(), bins=20, color="tab:blue", alpha=0.75)
plt.axvline(non_zero_counts.to_numpy().flatten().min(), color="grey", ls="--")
plt.axvline(non_zero_counts.to_numpy().flatten().max(), color="grey", ls="--")
plt.xlabel("Non-zero count")
plt.ylabel("Frequency");

In [ ]:
# First and last observations for each client

def get_min_max_timestamps_by_client(uci_df: pl.DataFrame) -> pl.DataFrame:
    min_max_ts = (
        uci_df.unpivot(
            on=[c for c in UCI_DF.columns if c != "timestamp"],
            index="timestamp",
            variable_name="client"
        )
        .filter(pl.col("value") > 0)
        .group_by("client", maintain_order=True)
        .agg(
            min_timestamp=pl.col("timestamp").min(),
            max_timestamp=pl.col("timestamp").max()
        )
    )
    return min_max_ts


min_max_ts_by_client = get_min_max_timestamps_by_client(UCI_DF)

# Plot
first_ts = min_max_ts_by_client["min_timestamp"].to_list()
last_ts = min_max_ts_by_client["max_timestamp"].to_list()

fig, ax = plt.subplots(1, 1)
ax.scatter(first_ts, np.arange(len(first_ts)), alpha=0.1, color="tab:blue", label="first")
ax.scatter(last_ts, np.arange(len(last_ts)), alpha=0.1, color="tab:red", label="last")
ax.legend()
for tick in ax.get_xticklabels():
    tick.set_rotation(45)
ax.set(xlabel="Timestamp", ylabel="Client")
fig.align_labels()
fig.tight_layout();

In [ ]:
# Are timestamps continuous?

def get_non_continuous_timeseries_by_client(uci_df: pl.DataFrame) -> dict[str, pl.DataFrame]:
    non_cont_ts: dict[str, pl.DataFrame] = {}

    for col in uci_df.columns:
        if col == "timestamp":
            continue
        
        result = (
            # select timestamp and col columns
            uci_df.select(pl.col("timestamp"), pl.col(col))
            # filter for where col > 0, sort by timestamp
            .filter(pl.col(col) > 0).sort(by="timestamp")
            # add column for first diff on timestamp
            .with_columns(pl.col("timestamp").diff().alias("timestamp_diff"))
            # filer result to where timestamp_diff > FREQUENCY
            .filter(pl.col("timestamp_diff").dt.total_seconds() > FREQUENCY_MINUTES * 60)
        )

        if result.is_empty():
            continue

        non_cont_ts[col] = result
    
    return non_cont_ts

non_continuous_timeseries = get_non_continuous_timeseries_by_client(UCI_DF)


In [ ]:
# Get proportion of non-continuous timestamps as a function of total
# length of timeseries

pct_non_continous: dict[str, tuple[float, float]] = {}
for client, non_cont_client_df in non_continuous_timeseries.items():
    # Get min / max timestamps for this client
    client_min_max_ts = min_max_ts_by_client.filter(pl.col("client") == client)
    [client_min_ts] = client_min_max_ts["min_timestamp"].to_list()
    [client_max_ts] = client_min_max_ts["max_timestamp"].to_list()
    
    expected_client_ts = pl.datetime_range(
        start=client_min_ts,
        end=client_max_ts,
        interval=f"{FREQUENCY_MINUTES}m",
        closed="both",
        eager=True,   
    )
    n_expected_client_ts = len(expected_client_ts)

    # Calculate number of non-continuous timestamps
    [n_non_cont_client_ts] = (
        non_cont_client_df
        # Calculate number of freq multiples of timestamp diff
        # Subtract 1.0 since a multiple of 1.0 corresponds to "cont" timestamp diff
        .select(freq_multiple=(pl.col("timestamp_diff") / pl.duration(minutes=FREQUENCY_MINUTES)) - 1.0)
        # Sum across all timestamps
        .sum()
        ["freq_multiple"]
        .to_list()
    )
    
    pct_non_continous[client] = (n_non_cont_client_ts, n_expected_client_ts)

In [ ]:
# Plot

non_cont_clients_and_pct = [
    (client, (n / d))
    for (client, (n, d)) in pct_non_continous.items()
]
non_cont_clients_and_pct = sorted(non_cont_clients_and_pct, key=itemgetter(1))

non_cont_client_names, non_cont_client_pcts = zip(*non_cont_clients_and_pct)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

# Distribution of proportion of non cont timestamps
axes[0].hist(non_cont_client_pcts, bins=50, alpha=0.5);
axes[0].set(xlabel="Fraction Non Continuous Timestamps", ylabel="Counts")

# Scatter plot by client id for pct > 1%
large_non_cont_pct = [(c, pct) for (c, pct) in non_cont_clients_and_pct if pct > 0.005]
large_non_cont_client_names, large_non_cont_client_pcts = zip(*large_non_cont_pct)
axes[1].scatter(large_non_cont_client_names, large_non_cont_client_pcts, alpha=0.5);
axes[1].set(ylabel="Fraction Non Continuous Timestamps")
for tick in axes[1].get_xticklabels():
    tick.set_rotation(90)

fig.tight_layout();

### Local EDA

In [ ]:
from constants import UCI_CLIENT_SITES_TO_VALIDATE

In [ ]:
# Load preprocessed data

uci_df = pl.read_parquet(UCI_DATA_DIR / "preprocessed.pq")
UCI_CLIENT_DF = uci_df.filter(pl.col("client").is_in(UCI_CLIENT_SITES_TO_VALIDATE))

### Trend & Seasonality

In [ ]:
fig, ax = plt.subplots(4, 1, figsize=(15, 10), sharex=False)

year = 2014
quarter = [
    (datetime(year, 1, 1), datetime(year, 4, 1)),
    (datetime(year, 4, 1), datetime(year, 7, 1)),
    (datetime(year, 7, 1), datetime(year, 10, 1)),
    (datetime(year, 10, 1), datetime(year + 1, 1, 1)),
]

for idx, (start_ts, end_ts) in enumerate(quarter):
    
    # Consumption data for each individual client for year.
    for client in UCI_CLIENT_DF["client"].unique().to_list():
        client_df = (
            UCI_CLIENT_DF
            .filter(
                pl.col("timestamp").is_between(start_ts, end_ts, closed="left"),
                pl.col("client") == client
            )
            .select(pl.col("timestamp"), pl.col("demand"))
            .sort(by="timestamp")
        )
        ax[idx].plot(
            client_df["timestamp"].to_list(),
            client_df["demand"].to_list(),
            alpha=0.5,
            color="grey",
            lw=0.75
        )

    # Mean consumption across all sites.
    mean_demand = (
        UCI_CLIENT_DF
        .filter(pl.col("timestamp").is_between(start_ts, end_ts, closed="left"))
        .group_by(pl.col("timestamp"))
        .agg(demand_mean=pl.col("demand").mean())
        .sort(by="timestamp")
    )
    ax[idx].plot(
        mean_demand["timestamp"].to_list(),
        mean_demand["demand_mean"].to_list(),
        label="Mean",
        color="tab:blue",
        lw=1.5,
    )
    
    ax[idx].set(ylabel="Demand (kW)")

fig.suptitle(f"Demand data for year {year}")
fig.tight_layout();

In [ ]:
clients = sorted(UCI_CLIENT_DF["client"].unique().to_list())

year = 2014
year_start, year_end = datetime(year, 1, 1), datetime(year + 1, 1, 1)
hourly_timestamps = pl.datetime_range(
    start=year_start,
    end=year_end,
    interval="1h",
    eager=True,
    closed="left"
)

n_cols = 5
n_rows = math.ceil(len(clients) // n_cols)

cmap = plt.cm.managua
vmin, vmax = -0.1, 0.9

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.5, n_rows * 3), sharex=True)
axes = axes.flatten()

for idx, client in enumerate(clients):
    client_df = (
        UCI_CLIENT_DF
        .filter(
            pl.col("client") == client,
            pl.col("timestamp").is_between(year_start, year_end, closed="left")
        )
        .join(
            other=hourly_timestamps.to_frame("timestamp"),
            on="timestamp",
            how="inner"
        )
        .with_columns(
            minute=pl.col("timestamp").dt.minute(),
            hour=pl.col("timestamp").dt.hour(),
            minute_of_day=pl.col("timestamp").dt.hour().cast(pl.Int32) * 60 + pl.col("timestamp").dt.minute().cast(pl.Int32),
            weekday=pl.col("timestamp").dt.weekday(),
        )
        .sort(by="timestamp")
    )

    # Scatter plot of demand vs hour colored by weekday
    weekdays = client_df["weekday"].unique().to_list()
    for weekday in weekdays:
        weekday_df = client_df.filter(pl.col("weekday") == weekday)
        mean_hourly_demand = weekday_df.group_by("hour").agg(demand_mean=pl.col("demand").mean()).sort(by="hour")
        
        color_val = vmin + (vmax - vmin) * weekday / len(weekdays)
        axes[idx].scatter(
            weekday_df["hour"].to_list(),
            weekday_df["demand"].to_list(),
            alpha=0.35,
            s=10,
            color=cmap(color_val),
        )
        axes[idx].plot(
            mean_hourly_demand["hour"].to_list(),
            mean_hourly_demand["demand_mean"].to_list(),
            color=cmap(color_val),
            lw=2.0,
            label=f"Weekday={weekday}",
        )
        axes[idx].set(title=client)
    
    if idx % n_cols == 0:
        axes[idx].set(ylabel="Demand (kW)")
    if idx // n_cols == n_rows - 1:
        axes[idx].set(xlabel="Hour of Day")

# Shared legend centered above the plot
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=len(clients), bbox_to_anchor=(0.8, 1.02))
fig.align_labels()
fig.tight_layout();

In [ ]:
# FFT spectra

clients = sorted(UCI_CLIENT_DF["client"].unique().to_list())

n_cols = 5  
n_rows = math.ceil(len(clients) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.5, n_rows * 3), sharex=True)
axes = axes.flatten()

for idx, client in enumerate(clients):
    client_df = (
        UCI_CLIENT_DF
        .filter(pl.col("client") == client)
        .sort(by="timestamp")
    )

    demand_data = client_df["demand"].to_numpy()
    demand_data_scaled = demand_data - demand_data.mean()
    demand_fft = np.fft.rfft(demand_data_scaled)

    # Sample spacing is 15 mins or 1/4 of an hour
    freq_fft = np.fft.rfftfreq(len(demand_data), d=0.25)
    mask = freq_fft < 0.5
    axes[idx].plot(
        freq_fft[mask],
        np.abs(demand_fft[mask]),
        color=plt.cm.berlin(0.5),
    )

    # Plot a vertical line at the frequency of the FFT
    daily_freq = 1 / 24
    axes[idx].axvline(daily_freq, color="tab:red", ls="--", lw=1.5, label="Daily")
    
    weekly_freq = 1 / (24 * 7)
    axes[idx].axvline(weekly_freq, color="tab:blue", ls="--", lw=1.5, label="Weekly")
    
    axes[idx].legend()
    axes[idx].set(title=client, yticklabels=[])
    axes[idx].grid(which="both", ls="--", alpha=0.5, c="grey", lw=0.5)
    if idx // n_cols == n_rows - 1:
        axes[idx].set(xlabel="Frequency (1/hours)")
    if idx % n_cols == 0:
        axes[idx].set(ylabel="FFT Magnitude")


fig.tight_layout();


In [ ]:
# What patterns remain if we remove the daily variation by plotting the average
# consumption for each day?

clients = sorted(UCI_CLIENT_DF["client"].unique().to_list())

years = [
    (datetime(2011, 1, 1), datetime(2012, 1, 1), "2011"),
    (datetime(2012, 1, 1), datetime(2013, 1, 1), "2012"),
    (datetime(2013, 1, 1), datetime(2014, 1, 1), "2013"),
    (datetime(2014, 1, 1), datetime(2015, 1, 1), "2014"),
]
normalize_year = 2000

fig, ax = plt.subplots(len(clients), 1, figsize=(15, len(clients) * 1.75), sharex=True)

cmap = plt.cm.managua
vmin, vmax = -0.1, 0.9

for idx, client in enumerate(clients):
    for year_idx, (start_ts, end_ts, year_label) in enumerate(years):
        # Consumption data for each individual client for year.
        daily_average_demand = (
            UCI_CLIENT_DF
            .filter(
                pl.col("timestamp").is_between(start_ts, end_ts, closed="left"),
                pl.col("client") == client
            )
            .group_by(pl.col("timestamp").dt.truncate(every="1d"))
            .agg(demand_mean=pl.col("demand").mean())
            .sort(by="timestamp")
        )
        
        # Normalize dates to common year (remove year component)
        normalized_dates = [
            datetime(normalize_year, ts.month, ts.day)
            for ts in daily_average_demand["timestamp"].to_list()
        ]
        
        color_val = vmin + (vmax - vmin) * year_idx / len(years)
        ax[idx].plot(
            normalized_dates,
            daily_average_demand["demand_mean"].to_list(),
            label=year_label,
            color=cmap(color_val),
            lw=2.5,
        )
    
    # Format x-axis to show month-day labels
    ax[idx].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    ax[idx].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    ax[idx].xaxis.set_minor_locator(mdates.MonthLocator())
    ax[idx].set(ylabel="Demand (kW)", title=client)
    ax[idx].grid(which="both", ls="--", alpha=0.5, c="grey", lw=0.5)


handles, labels = ax[0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=len(clients), bbox_to_anchor=(1.0, 1.001))
fig.align_labels()    
fig.tight_layout();

In [ ]:
# TODO: Proper trend-seasonal decomposition

In [ ]:
# Distribution plots; one for each client showing the distribution of demand for each year

clients = sorted(UCI_CLIENT_DF["client"].unique().to_list())

years = [
    (datetime(2011, 1, 1), datetime(2012, 1, 1), "2011"),
    (datetime(2012, 1, 1), datetime(2013, 1, 1), "2012"),
    (datetime(2013, 1, 1), datetime(2014, 1, 1), "2013"),
    (datetime(2014, 1, 1), datetime(2015, 1, 1), "2014"),
]

n_cols = 5
n_rows = math.ceil(len(clients) // n_cols)

cmap = plt.cm.managua
vmin, vmax = -0.1, 1.1

fig, ax = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.5, n_rows * 3))
ax = ax.flatten()

for idx, client in enumerate(clients):
    for year_idx, (start_ts, end_ts, year_label) in enumerate(years):
        client_df = (
            UCI_CLIENT_DF
            .filter(
                pl.col("timestamp").is_between(start_ts, end_ts, closed="left"),
                pl.col("client") == client
            )
        )
        if client_df.is_empty():
            continue
        
        color_val = vmin + (vmax - vmin) * year_idx / len(years)
        ax[idx].hist(
            client_df["demand"].to_list(),
            alpha=0.25,
            bins=50,
            color=cmap(color_val),
            density=True,
            # label=year_label,
        )

        # Add KDE line
        kde = gaussian_kde(client_df["demand"].to_list())
        x = np.linspace(0, client_df["demand"].max(), 1000)
        ax[idx].plot(x, kde(x), color=cmap(color_val), lw=2.25, label=year_label)

        ax[idx].set(title=client)

    if idx % n_cols == 0:
        ax[idx].set(ylabel="Density")
    if idx // n_cols == n_rows - 1:
        ax[idx].set(xlabel="Demand (kW)")

handles, labels = ax[-1].get_legend_handles_labels()
fig.legend(handles, labels, ncol=len(clients), bbox_to_anchor=(1.0, 1.02))

fig.align_labels()
fig.tight_layout();

### Stationarity and Autocorrelation

In [ ]:
# Seasonally differenced plots

clients = sorted(UCI_CLIENT_DF["client"].unique().to_list())

years = [
    (datetime(2011, 1, 1), datetime(2012, 1, 1), "2011"),
    (datetime(2012, 1, 1), datetime(2013, 1, 1), "2012"),
    (datetime(2013, 1, 1), datetime(2014, 1, 1), "2013"),
    (datetime(2014, 1, 1), datetime(2015, 1, 1), "2014"),
]
normalize_year = 2000

fig, ax = plt.subplots(len(clients), 1, figsize=(15, len(clients) * 1.75), sharex=True)

cmap = plt.cm.managua
vmin, vmax = -0.1, 1.1

for idx, client in enumerate(clients):
    ax[idx].axhline(0, color="black", lw=0.5)
    for year_idx, (start_ts, end_ts, year_label) in enumerate(years):
        seasonal_diff_demand = (
            UCI_CLIENT_DF
            .filter(
                pl.col("client") == client,
                pl.col("timestamp").is_between(start_ts, end_ts, closed="left")
            )
            .sort(by="timestamp")
            .with_columns(demand_diff=pl.col("demand").diff(24 / (FREQUENCY_MINUTES / 60)))
            .drop_nulls(subset=["demand_diff"])
        )

        # Normalize dates to common year (remove year component)
        normalized_dates = [
            datetime.replace(ts, year=normalize_year)
            for ts in seasonal_diff_demand["timestamp"].to_list()
        ]

        color_val = vmin + (vmax - vmin) * year_idx / len(years)
        ax[idx].plot(
            normalized_dates,
            seasonal_diff_demand["demand_diff"].to_list(),
            label=year_label,
            color=cmap(color_val),
            lw=1.0,
            alpha=0.75,
        )
    
    # Format x-axis to show month-day labels
    ax[idx].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    ax[idx].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    ax[idx].xaxis.set_minor_locator(mdates.MonthLocator())
    ax[idx].set(ylabel="Demand (kW)", title=client)


handles, labels = ax[0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=len(clients), bbox_to_anchor=(1.0, 1.001))
fig.align_labels()    
fig.tight_layout();


In [ ]:
# Autocorrelation plots

clients = sorted(UCI_CLIENT_DF["client"].unique().to_list())

fig, ax = plt.subplots(len(clients), 2, figsize=(15, len(clients) * 1.75), sharex=True)

cmap = plt.cm.managua
vmin, vmax = -0.1, 1.1

for idx, client in enumerate(clients):
    seasonal_diff_demand = (
        UCI_CLIENT_DF
        .filter(
            pl.col("client") == client,
            pl.col("timestamp").is_between(start_ts, end_ts, closed="left")
        )
        .sort(by="timestamp")
        .with_columns(demand_diff=pl.col("demand").diff(24 / (FREQUENCY_MINUTES / 60)))
        .drop_nulls(subset=["demand_diff"])
    )

    if seasonal_diff_demand.is_empty():
        continue

    # Plot autocorrelation
    s = seasonal_diff_demand.select(pl.col("timestamp"), pl.col("demand_diff")).to_pandas().set_index("timestamp")
    color = plt.cm.managua(0.6)
    plot_acf(
        s["demand_diff"],
        lags=2 * (24 / (FREQUENCY_MINUTES / 60)),
        ax=ax[idx, 0],
        title="Autocorrelation: " + client,
        color=color,
        markersize=3,
        vlines_kwargs={"color": color},
    );
    plot_pacf(
        s["demand_diff"],
        lags=2 * (24 / (FREQUENCY_MINUTES / 60)),
        ax=ax[idx, 1],
        title="Partial Autocorrelation: " + client,
        color=color,
        markersize=3,
        vlines_kwargs={"color": color},
    );
    ax[idx, 0].set(ylabel="Correlation")

fig.align_labels()    
fig.tight_layout();

In [ ]:
# Distribution plots of the seasonally differenced data

clients = sorted(UCI_CLIENT_DF["client"].unique().to_list())

years = [
    (datetime(2011, 1, 1), datetime(2012, 1, 1), "2011"),
    (datetime(2012, 1, 1), datetime(2013, 1, 1), "2012"),
    (datetime(2013, 1, 1), datetime(2014, 1, 1), "2013"),
    (datetime(2014, 1, 1), datetime(2015, 1, 1), "2014"),
]

n_cols = 5
n_rows = math.ceil(len(clients) // n_cols)

# Define color map and min/max to exclude extreme colors
cmap = plt.cm.managua
vmin, vmax = 0.0, 0.9

fig, ax = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.5, n_rows * 3))
ax = ax.flatten()

for idx, client in enumerate(clients):
    for year_idx, (start_ts, end_ts, year_label) in enumerate(years):
        seasonal_diff_demand = (
            UCI_CLIENT_DF
            .filter(
                pl.col("client") == client,
                pl.col("timestamp").is_between(start_ts, end_ts, closed="left")
            )
            .sort(by="timestamp")
            .with_columns(demand_diff=pl.col("demand").diff(int(24 / (FREQUENCY_MINUTES / 60))))
            .drop_nulls(subset=["demand_diff"])
        )

        if seasonal_diff_demand.is_empty():
            continue
        
        color_val = vmin + (vmax - vmin) * year_idx / len(years)
        ax[idx].hist(
            seasonal_diff_demand["demand_diff"].to_list(),
            bins=50,
            alpha=0.25,
            color=cmap(color_val),
            density=True,
        )

        # Add KDE line
        kde = gaussian_kde(seasonal_diff_demand["demand_diff"].to_list())
        x = np.linspace(seasonal_diff_demand["demand_diff"].min(), seasonal_diff_demand["demand_diff"].max(), 1000)
        ax[idx].plot(x, kde(x), color=cmap(color_val), lw=2.25, label=year_label)

        ax[idx].set(title=client)

    if idx % n_cols == 0:
        ax[idx].set(ylabel="Density")
    if idx // n_cols == n_rows - 1:
        ax[idx].set(xlabel="Demand (kW)")

handles, labels = ax[-1].get_legend_handles_labels()
fig.legend(handles, labels, ncol=len(clients), bbox_to_anchor=(1.0, 1.02))

fig.align_labels()
fig.tight_layout();